In [ ]:
"""
Análisis Tecno-Económico del Agregador en España
Extracción y Visualización de la "Curva del Pato" (Demanda Neta vs Generación Solar)

Este script consulta la API de ESIOS (Red Eléctrica de España) para extraer
series temporales reales de demanda eléctrica y generación solar fotovoltaica,
calculando la demanda neta del sistema para un día específico.

Autor: Alberto Zafra Muñoz
"""

import os
import sys
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ==========================================
# 1. CONFIGURACIÓN DE LA API Y PARÁMETROS
# ==========================================
# SEGURIDAD: Nunca introducir el token real en el código fuente. 
# Se recomienda usar variables de entorno (os.getenv).
TOKEN = os.getenv('ESIOS_TOKEN', 'INTRODUCE_TU_TOKEN_AQUI')

HEADERS = {
    'Accept': 'application/json; application/vnd.esios-api-v1+json',
    'Content-Type': 'application/json',
    'Host': 'api.esios.ree.es',
    'x-api-key': TOKEN
}

# Parámetros temporales del estudio
FECHA_INICIO = '2024-05-05T00:00:00+02:00'
FECHA_FIN = '2024-05-05T23:59:59+02:00'

# Identificadores oficiales de ESIOS
IND_DEMANDA = 1293  # Demanda Real
IND_SOLAR = 2044    # Generación Solar Fotovoltaica

# ==========================================
# 2. FUNCIONES DE EXTRACCIÓN DE DATOS
# ==========================================
def obtener_datos_esios(indicador: int, start: str, end: str, headers: dict) -> pd.Series:
    """
    Realiza una petición a la API de ESIOS para un indicador específico.

    Parámetros:
        indicador (int): Código del indicador de REE.
        start (str): Fecha de inicio en formato ISO 8601.
        end (str): Fecha de fin en formato ISO 8601.
        headers (dict): Cabeceras de autenticación para la petición HTTP.

    Retorna:
        pd.Series: Serie temporal con datos horarios promediados. 
                   Devuelve una serie vacía en caso de error.
    """
    url = f"https://api.esios.ree.es/indicators/{indicador}"
    params = {'start_date': start, 'end_date': end}

    try:
        response = requests.get(url, headers=headers, params=params)

        if response.status_code == 200:
            datos = response.json()
            valores = datos.get('indicator', {}).get('values', [])
            
            if not valores:
                print(f"[ADVERTENCIA] No hay datos para el indicador {indicador} en las fechas solicitadas.")
                return pd.Series(dtype=float)

            df = pd.DataFrame(valores)
            df['datetime'] = pd.to_datetime(df['datetime'])
            df.set_index('datetime', inplace=True)
            
            # Promediado horario para estandarizar la resolución temporal
            df_horario = df['value'].resample('h').mean()
            return df_horario
        else:
            print(f"[ERROR] Código HTTP {response.status_code} al consultar el indicador {indicador}.")
            return pd.Series(dtype=float)
            
    except requests.exceptions.RequestException as e:
        print(f"[ERROR DE CONEXIÓN] Fallo al contactar con la API de ESIOS: {e}")
        return pd.Series(dtype=float)

# ==========================================
# 3. EJECUCIÓN PRINCIPAL Y PROCESAMIENTO
# ==========================================
if __name__ == "__main__":
    print("Iniciando descarga de datos (Demanda Real)...")
    demanda_total = obtener_datos_esios(IND_DEMANDA, FECHA_INICIO, FECHA_FIN, HEADERS)

    print("Iniciando descarga de datos (Generación Solar Fotovoltaica)...")
    generacion_solar = obtener_datos_esios(IND_SOLAR, FECHA_INICIO, FECHA_FIN, HEADERS)

    # Verificación de integridad de los datos
    if demanda_total.empty or generacion_solar.empty:
        print("\n[!] ABORTANDO: Imposible proceder sin los conjuntos de datos completos.")
        sys.exit(1)

    # Consolidación de datos en un DataFrame unificado
    df_final = pd.concat([demanda_total, generacion_solar], axis=1, keys=['Demanda', 'Solar']).fillna(0)

    # Conversión de unidades (MW a GW) y cálculo del balance neto
    df_final['Demanda_GW'] = df_final['Demanda'] / 1000
    df_final['Solar_GW'] = df_final['Solar'] / 1000
    df_final['Demanda_Neta_GW'] = df_final['Demanda_GW'] - df_final['Solar_GW']

    # ==========================================
    # 4. REPRESENTACIÓN GRÁFICA
    # ==========================================
    plt.figure(figsize=(12, 6))
    plt.style.use('seaborn-v0_8-whitegrid')

    horas = df_final.index

    # Trazado de series
    plt.plot(horas, df_final['Demanda_GW'], color='gray', linestyle='--', linewidth=2, label='Demanda Total Real')
    plt.plot(horas, df_final['Demanda_Neta_GW'], color='#2ca02c', linewidth=3, label='Demanda Neta (La "Curva del Pato")')

    # Sombreado del área correspondiente al aporte solar
    plt.fill_between(horas, df_final['Demanda_Neta_GW'], df_final['Demanda_GW'],
                     color='#ff7f0e', alpha=0.3, label='Generación Solar FV (Vertida a red)')

    # Formateo del eje temporal
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=2))
    plt.gcf().autofmt_xdate(rotation=0, ha='center')

    # Etiquetas y Leyendas
    plt.title(f'Demanda Neta y Generación Solar en España ({FECHA_INICIO[:10]})', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Hora del día', fontsize=12)
    plt.ylabel('Potencia (GW)', fontsize=12)

    plt.ylim(0, df_final['Demanda_GW'].max() + 5)
    plt.xlim(horas.min(), horas.max())

    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=True)
    plt.tight_layout()
    plt.show()